# Baseline PCA Statistical Arbitrage

This is the **simple baseline** from our earlier 30.9% out-of-sample result.

### Strategy

**Multi-sector stocks → PCA residuals → mean-reversion z-score → cross-sectional StatArb → backtest**

No KL divergence, HMM, Wasserstein distance, BOCPD, LQR, dynamic factor selection, or other regime models.


## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import adfuller

np.random.seed(42)


## 2. Settings and stock universe

In [ ]:
START = "2010-01-01"
END = "2021-01-01"       # exclusive, so 2020 is included

TRAIN_RATIO = 0.90

N_FACTORS = 8
ROLLING_Z_WINDOW = 60

ADF_PVALUE_MAX = 0.10
MIN_HALF_LIFE = 2
MAX_HALF_LIFE = 120
MAX_TRADABLE_ASSETS = 20

ENTRY_Z = 1.0
N_LONG_SHORT = 3

TRANSACTION_COST_BPS = 5
INITIAL_CAPITAL = 100_000

SECTOR_MAP = {
    "Technology": ["AAPL", "MSFT", "IBM", "CSCO"],
    "Financials": ["JPM", "BAC", "GS", "C"],
    "Healthcare": ["JNJ", "PFE", "MRK", "ABT"],
    "Consumer Staples": ["PG", "KO", "PEP", "WMT"],
    "Consumer Discretionary": ["AMZN", "HD", "MCD", "NKE"],
    "Industrials": ["CAT", "MMM", "HON", "GE"],
    "Energy": ["XOM", "CVX", "COP", "SLB"],
    "Communication Services": ["T", "VZ", "DIS", "CMCSA"],
    "Utilities": ["SO", "DUK", "NEE", "EXC"],
    "Materials": ["LIN", "NUE", "APD", "FCX"],
    "Real Estate": ["AMT", "SPG", "PLD", "O"],
}

TICKERS = [
    ticker
    for tickers in SECTOR_MAP.values()
    for ticker in tickers
]

print(f"Universe: {len(TICKERS)} stocks across {len(SECTOR_MAP)} sectors")


## 3. Download prices and create the chronological 90/10 split

In [ ]:
raw = yf.download(
    TICKERS,
    start=START,
    end=END,
    auto_adjust=True,
    progress=False,
    threads=True,
)

prices = raw["Close"].copy()

if isinstance(prices, pd.Series):
    prices = prices.to_frame()

prices = prices.reindex(columns=TICKERS)

# Keep stocks with sufficiently complete histories.
coverage = prices.notna().mean()
prices = prices.loc[
    :,
    coverage[coverage >= 0.95].index
]

prices = prices.ffill().dropna()

returns = prices.pct_change().dropna()

split_idx = int(len(returns) * TRAIN_RATIO)

train_returns = returns.iloc[:split_idx]
test_returns = returns.iloc[split_idx:]

print(
    f"Training: {train_returns.index[0].date()} -> "
    f"{train_returns.index[-1].date()}"
)

print(
    f"Testing:  {test_returns.index[0].date()} -> "
    f"{test_returns.index[-1].date()}"
)


## 4. Fit PCA on training data and construct residuals

In [ ]:
def fit_pca(train_returns, n_factors):
    scaler = StandardScaler()

    X = scaler.fit_transform(train_returns)

    n_factors = min(
        n_factors,
        X.shape[1] - 1,
    )

    pca = PCA(
        n_components=n_factors,
        random_state=42,
    )

    pca.fit(X)

    return scaler, pca


def get_residuals(returns, scaler, pca):
    X = scaler.transform(returns)

    common_component = pca.inverse_transform(
        pca.transform(X)
    )

    residuals = pd.DataFrame(
        X - common_component,
        index=returns.index,
        columns=returns.columns,
    )

    return residuals


pca_scaler, pca_model = fit_pca(
    train_returns,
    N_FACTORS,
)

all_returns = pd.concat([
    train_returns,
    test_returns,
])

residual_returns = get_residuals(
    all_returns,
    pca_scaler,
    pca_model,
)

train_residual_returns = residual_returns.loc[
    train_returns.index
]

print(
    f"Factors used: {pca_model.n_components_}"
)

print(
    f"Variance explained: "
    f"{pca_model.explained_variance_ratio_.sum():.2%}"
)


## 5. Select mean-reverting PCA residuals

Residuals are screened using only the training period.

We use:
- Augmented Dickey–Fuller p-value
- estimated half-life

The strongest eligible residuals are selected for trading.


In [ ]:
def half_life(series):
    x = series.dropna().values

    if len(x) < 50:
        return np.nan

    lagged = x[:-1]
    delta = np.diff(x)

    X = np.column_stack([
        np.ones(len(lagged)),
        lagged,
    ])

    beta = np.linalg.lstsq(
        X,
        delta,
        rcond=None,
    )[0][1]

    if beta >= 0:
        return np.inf

    return float(
        -np.log(2) / beta
    )


def select_residuals(train_residual_returns):
    spreads = train_residual_returns.cumsum()

    rows = []

    for ticker in spreads.columns:
        spread = spreads[ticker]

        try:
            _, p_value, *_ = adfuller(
                spread.dropna(),
                autolag="AIC",
            )
        except Exception:
            p_value = np.nan

        rows.append({
            "ticker": ticker,
            "adf_pvalue": p_value,
            "half_life": half_life(spread),
        })

    stats = pd.DataFrame(rows).set_index("ticker")

    eligible = stats[
        (stats["adf_pvalue"] <= ADF_PVALUE_MAX)
        & (stats["half_life"] >= MIN_HALF_LIFE)
        & (stats["half_life"] <= MAX_HALF_LIFE)
    ]

    # Keep the previous baseline's fallback so the model
    # remains usable if the strict screen is too restrictive.
    if len(eligible) < min(
        8,
        MAX_TRADABLE_ASSETS,
    ):
        eligible = (
            stats
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=["adf_pvalue"])
            .sort_values(
                ["adf_pvalue", "half_life"]
            )
            .head(MAX_TRADABLE_ASSETS)
        )

    selected = (
        eligible
        .sort_values(
            ["adf_pvalue", "half_life"]
        )
        .head(MAX_TRADABLE_ASSETS)
    )

    return stats, selected


residual_stats, selected = select_residuals(
    train_residual_returns
)

selected_assets = selected.index.tolist()

print(
    f"Selected residuals: {len(selected_assets)}"
)

display(selected)


## 6. Generate the baseline StatArb signal

In [ ]:
spreads = residual_returns[
    selected_assets
].cumsum()

rolling_mean = spreads.rolling(
    ROLLING_Z_WINDOW,
    min_periods=ROLLING_Z_WINDOW,
).mean()

rolling_std = spreads.rolling(
    ROLLING_Z_WINDOW,
    min_periods=ROLLING_Z_WINDOW,
).std()

z_scores = (
    spreads - rolling_mean
) / rolling_std.replace(
    0,
    np.nan,
)


def make_stat_arb_weights(
    z_scores,
    entry_z,
    n_long_short,
):
    weights = pd.DataFrame(
        0.0,
        index=z_scores.index,
        columns=z_scores.columns,
    )

    for date, row in z_scores.iterrows():
        clean = row.dropna()

        if len(clean) < 2 * n_long_short:
            continue

        longs = (
            clean[clean <= -entry_z]
            .nsmallest(n_long_short)
        )

        shorts = (
            clean[clean >= entry_z]
            .nlargest(n_long_short)
        )

        if len(longs) > 0:
            weights.loc[
                date,
                longs.index,
            ] = 1.0 / len(longs)

        if len(shorts) > 0:
            weights.loc[
                date,
                shorts.index,
            ] = -1.0 / len(shorts)

    return weights


weights = make_stat_arb_weights(
    z_scores,
    ENTRY_Z,
    N_LONG_SHORT,
)

print("Baseline StatArb signal ready.")


## 7. Backtest and out-of-sample performance

In [ ]:
def backtest(
    weights,
    returns,
    initial_capital,
    cost_bps,
):
    common = weights.index.intersection(
        returns.index
    )

    w = weights.loc[common].fillna(0.0)

    r = returns.loc[
        common,
        w.columns,
    ].fillna(0.0)

    # Avoid look-ahead: today's P&L uses yesterday's weights.
    held_weights = w.shift(1).fillna(0.0)

    daily_return = (
        held_weights * r
    ).sum(axis=1)

    turnover = (
        w.diff()
        .abs()
        .sum(axis=1)
        .fillna(w.abs().sum(axis=1))
    )

    trading_cost = (
        turnover
        * cost_bps
        / 10_000
    )

    net_return = (
        daily_return
        - trading_cost
    )

    equity = (
        initial_capital
        * (1 + net_return).cumprod()
    )

    return (
        net_return,
        equity,
        turnover,
        trading_cost,
    )


def performance_stats(
    pnl,
    equity,
    turnover,
    trading_cost,
):
    years = max(
        len(pnl) / 252,
        1 / 252,
    )

    cagr = (
        equity.iloc[-1]
        / equity.iloc[0]
    ) ** (1 / years) - 1

    volatility = (
        pnl.std()
        * np.sqrt(252)
    )

    sharpe = (
        pnl.mean()
        / pnl.std()
        * np.sqrt(252)
        if pnl.std() > 0
        else np.nan
    )

    drawdown = (
        equity
        / equity.cummax()
        - 1
    )

    return pd.Series({
        "Total Return": (
            equity.iloc[-1]
            / equity.iloc[0]
            - 1
        ),
        "CAGR": cagr,
        "Annual Volatility": volatility,
        "Sharpe": sharpe,
        "Max Drawdown": drawdown.min(),
        "Total Turnover": turnover.sum(),
        "Trading Cost": trading_cost.sum(),
    })


test_weights = weights.reindex(
    test_returns.index
).fillna(0.0)

test_asset_returns = returns.loc[
    test_returns.index,
    selected_assets,
]

pnl, equity, turnover, trading_cost = backtest(
    test_weights,
    test_asset_returns,
    INITIAL_CAPITAL,
    TRANSACTION_COST_BPS,
)

results = performance_stats(
    pnl,
    equity,
    turnover,
    trading_cost,
)

display(
    results.to_frame(
        "Baseline PCA StatArb"
    )
)


## 8. Equity curve

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    equity.index,
    equity.values,
)

plt.title(
    "Baseline PCA StatArb — Out-of-Sample"
)
plt.xlabel("Date")
plt.ylabel("Portfolio Value")

plt.tight_layout()
plt.show()
